## Paso 1: Librerías y Configuración

In [1]:
import os
import pandas as pd
import sys
from pathlib import Path
current_dir = Path.cwd()
from collections import Counter, defaultdict

# Asegurar que el paquete local "sources" esté disponible
sources_dir = current_dir / 'sources'
if str(sources_dir) not in sys.path:
    sys.path.insert(0, str(sources_dir))

from prepro_func import ensure_nltk_resources, remove_special_characters, tokenize, stemming_tokens, build_tfidf_matrix, score_queries_tfidf
from bm25_model import build_bm25_index, bm25_score_doc, bm25_rank, score_queries_bm25

ensure_nltk_resources()

print("Librerías y módulos cargados correctamente")

Librerías y módulos cargados correctamente


## Paso 2: Carga de archivos y construcción del corpus

In [2]:
data_dir = Path.cwd() / 'data'
csv_files = sorted(data_dir.glob('*.csv'))

print(f"Archivos CSV encontrados: {len(csv_files)}")

# Cargar todos los CSVs
dataframes = []
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file, encoding='utf-8')
        print(f"✅ {csv_file.name}: {len(df)} filas cargadas")
        dataframes.append(df)
    except Exception as e:
        print(f"❌ Error cargando {csv_file.name}: {e}")

# Concatenar todos los dataframes
df_corpus = pd.concat(dataframes, ignore_index=True)

print(f"\nCorpus completo: {len(df_corpus)} documentos")
print(f"Columnas disponibles: {df_corpus.columns.tolist()}")

# Seleccionar columnas relevantes y usar description_final como texto principal
df_corpus = df_corpus[['job_id', 'job_title', 'company', 'careers_required', 'description_final']].copy()
df_corpus.rename(columns={'description_final': 'text'}, inplace=True)

# Verificar datos
print(f"\nCorpus preparado con columnas: {df_corpus.columns.tolist()}")
print(f"Primeras 2 filas (primeros 200 caracteres):")
df_corpus.head(2)

Archivos CSV encontrados: 24
✅ Administración_de_Empresas_Merged.csv: 5220 filas cargadas
✅ Agroindustria_Merged.csv: 1688 filas cargadas
✅ Ciencia_de_Datos_Merged.csv: 4836 filas cargadas
✅ Computación_Merged.csv: 4101 filas cargadas
✅ Economía_Merged.csv: 2506 filas cargadas
✅ Electricidad_Merged.csv: 2066 filas cargadas
✅ Electrónica_y_Automatización_Merged.csv: 6349 filas cargadas
✅ Física_Merged.csv: 1614 filas cargadas
✅ Geología_Merged.csv: 1074 filas cargadas
✅ Ingeniería_Ambiental_Merged.csv: 3295 filas cargadas
✅ Ingeniería_Civil_Merged.csv: 3864 filas cargadas
✅ Ingeniería_de_la_Producción_Merged.csv: 4503 filas cargadas
✅ Ingeniería_Química_Merged.csv: 1148 filas cargadas
✅ Inteligencia_Artificial_Merged.csv: 6805 filas cargadas
✅ Matemática_Aplicada_Merged.csv: 383 filas cargadas
✅ Matemática_Merged.csv: 399 filas cargadas
✅ Materiales_Merged.csv: 2127 filas cargadas
✅ Mecatrónica_Merged.csv: 1338 filas cargadas
✅ Mecánica_Merged.csv: 1633 filas cargadas
✅ Petróleos_Merged

,job_id,job_title,company,careers_required,text
0,69fc771c82ad86bd1f49714ab4c488afbf817a99d5702f...,Asistente de Administración y Gerencia experie...,HIALPESA,[],Importante empresa Textil con más de 40 años e...
1,4791ac695898b5787d37a855ad4409f4786898125a537c...,Docente JP Administración de empresas Tumbes,SENATI,[],DOCENTE EN ADMINISTRACIÓN Institución Líder y ...


## Paso 3: Limpieza de caracteres especiales

In [3]:
# Mostrar antes de limpiar
print("ANTES (200 caracteres):")
print(df_corpus['text'].iloc[0][:200]+"\n")

# Aplicar remove_special_characters
df_corpus['clean_text'] = df_corpus['text'].fillna('').apply(remove_special_characters)

print("DESPUÉS (200 caracteres):")
print(df_corpus['clean_text'].iloc[0][:200])
print("\n" + "="*80 + "\n")

# Estadísticas
print(f"✅ Limpieza completada")
print(f"Promedio caracteres antes: {df_corpus['text'].str.len().mean():.0f}")
print(f"Promedio caracteres después: {df_corpus['clean_text'].str.len().mean():.0f}")
print(f"\nMuestra de 3 documentos limpios:")
df_corpus[['job_title', 'clean_text']].head(3)

ANTES (200 caracteres):
Importante empresa Textil con más de 40 años en el mercado, dedicada a la fabricación y exportación de prendas de vestir se encuentra en busca de ASISTENTE ADMINISTRATIVO FINANCIERO Requisitos Bachill

DESPUÉS (200 caracteres):
Importante empresa Textil con más de 40 años en el mercado dedicada a la fabricación y exportación de prendas de vestir se encuentra en busca de ASISTENTE ADMINISTRATIVO FINANCIERO Requisitos Bachille


✅ Limpieza completada
Promedio caracteres antes: 714
Promedio caracteres después: 687

Muestra de 3 documentos limpios:


,job_title,clean_text
0,Asistente de Administración y Gerencia experie...,Importante empresa Textil con más de 40 años e...
1,Docente JP Administración de empresas Tumbes,DOCENTE EN ADMINISTRACIÓN Institución Líder y ...
2,Aprendiz Universitario administración de empr...,Nos encontramos en la búsqueda de un aprendiz ...


## Paso 4: Tokenización

In [4]:
# Aplicar tokenización en español
df_corpus['tokens'] = df_corpus['clean_text'].apply(
    lambda x: tokenize(x, language='spanish', remove_stopwords=True)
)

# Estadísticas
print(f"✅ Tokenización completada")
print(f"Promedio de tokens por documento: {df_corpus['tokens'].apply(len).mean():.0f}")
print(f"Máximo número de tokens: {df_corpus['tokens'].apply(len).max()}")
print(f"Mínimo número de tokens: {df_corpus['tokens'].apply(len).min()}")
print("\n")

print("Muestra de 3 documentos tokenizados:")
for i in range(3):
    print(f"\n Documento {i+1}: {df_corpus['job_title'].iloc[i]}")
    tokens_list = df_corpus['tokens'].iloc[i]
    print(f"   Primeros 15 tokens: {tokens_list[:15]}")
    print(f"   Total de tokens: {len(tokens_list)}")

✅ Tokenización completada
Promedio de tokens por documento: 60
Máximo número de tokens: 1582
Mínimo número de tokens: 0


Muestra de 3 documentos tokenizados:

 Documento 1: Asistente de Administración y Gerencia experiencia en empresas industriales
   Primeros 15 tokens: ['importante', 'empresa', 'textil', 'años', 'mercado', 'dedicada', 'fabricación', 'exportación', 'prendas', 'vestir', 'encuentra', 'busca', 'asistente', 'administrativo', 'financiero']
   Total de tokens: 78

 Documento 2: Docente JP Administración  de empresas Tumbes
   Primeros 15 tokens: ['docente', 'administración', 'institución', 'líder', 'prestigiosa', 'formación', 'profesional', 'apoya', 'industria', 'nacional', 'contexto', 'global', 'contribuye', 'mejora', 'calidad']
   Total de tokens: 158

 Documento 3: Aprendiz  Universitario administración de empresas  Etapa práctica
   Primeros 15 tokens: ['encontramos', 'búsqueda', 'aprendiz', 'estudiante', 'universitario', 'administración', 'empresas', 'convenio', 'sena

## Paso 5: Stemming 

In [5]:
# Aplicar stemming a los tokens
df_corpus['stemmed'] = df_corpus['tokens'].apply(
    lambda tokens: ' '.join(stemming_tokens(tokens, language='spanish'))
)

print(f"✅ Stemming completado")
print(f"Promedio de caracteres en texto stemmed: {df_corpus['stemmed'].str.len().mean():.0f}")

print("\n")

print("Comparación Tokens → Stemmed (3 ejemplos):\n")
for i in range(3):
    print(f" Documento {i+1}: {df_corpus['job_title'].iloc[i]}")
    tokens_list = df_corpus['tokens'].iloc[i][:10]
    print(f"   Tokens (primeros 10):  {tokens_list}")
    stemmed_list = df_corpus['stemmed'].iloc[i].split()[:10]
    print(f"   Stemmed (primeros 10): {stemmed_list}")
    print()

✅ Stemming completado
Promedio de caracteres en texto stemmed: 422


Comparación Tokens → Stemmed (3 ejemplos):

 Documento 1: Asistente de Administración y Gerencia experiencia en empresas industriales
   Tokens (primeros 10):  ['importante', 'empresa', 'textil', 'años', 'mercado', 'dedicada', 'fabricación', 'exportación', 'prendas', 'vestir']
   Stemmed (primeros 10): ['import', 'empres', 'textil', 'años', 'merc', 'dedic', 'fabric', 'export', 'prend', 'vest']

 Documento 2: Docente JP Administración  de empresas Tumbes
   Tokens (primeros 10):  ['docente', 'administración', 'institución', 'líder', 'prestigiosa', 'formación', 'profesional', 'apoya', 'industria', 'nacional']
   Stemmed (primeros 10): ['docent', 'administr', 'institu', 'lid', 'prestigi', 'formacion', 'profesional', 'apoy', 'industri', 'nacional']

 Documento 3: Aprendiz  Universitario administración de empresas  Etapa práctica
   Tokens (primeros 10):  ['encontramos', 'búsqueda', 'aprendiz', 'estudiante', 'universitario

## Paso 6: Índice invertido

In [6]:
# Construcción
df_corpus['tokens'] = df_corpus['tokens'].apply(lambda x: x if isinstance(x, list) else [])

inverted_index = defaultdict(lambda: {
    'doc_freq': 0,
    'total_freq': 0,
    'postings': []
})

for doc_idx, tokens in enumerate(df_corpus['tokens']):
    term_counts = Counter(tokens)
    for term, freq in term_counts.items():
        entry = inverted_index[term]
        entry['doc_freq'] += 1
        entry['total_freq'] += freq
        entry['postings'].append({
            'doc_index': doc_idx,
            'job_id': df_corpus['job_id'].iloc[doc_idx],
            'job_title': df_corpus['job_title'].iloc[doc_idx],
            'freq': freq
        })

# Convertir a diccionario normal para facilitar inspección
inverted_index = dict(inverted_index)

# Mostrar información general del índice
print(f"✅ Índice invertido construido")
print(f"Términos únicos en el índice: {len(inverted_index)}")

# Crear un DataFrame del índice invertido para mostrar estadísticas de los términos
summary_rows = []
for term in sorted(inverted_index):
    summary_rows.append({
        'term': term,
        'doc_freq': inverted_index[term]['doc_freq'],
        'total_freq': inverted_index[term]['total_freq']
    })
summary_df = pd.DataFrame(summary_rows)
print("\nÍndice invertido:")
display(summary_df)

# Mostrar el índice invertido de algunos términos de ejemplo
test_terms = list(summary_df['term'].head(5))
for term in test_terms:
    print(f"\nTérmino: '{term}'")
    print(f"  Documentos: {inverted_index[term]['doc_freq']}")
    print(f"  Frecuencia total: {inverted_index[term]['total_freq']}")
    print("  Posting list:")
    for posting in inverted_index[term]['postings'][:5]:
        print(f"    - doc_index={posting['doc_index']}, job_id={posting['job_id']}, freq={posting['freq']}, title={posting['job_title']}")


✅ Índice invertido construido
Términos únicos en el índice: 83775

Índice invertido:


,term,doc_freq,total_freq
0,aa,34,37
1,aaa,25,32
2,aaaradius,1,1
3,aaas,1,1
4,aace,1,1
...,...,...,...
83770,úpo,1,1
83771,úselos,2,2
83772,úsqueda,1,1
83773,útil,174,195



Término: 'aa'
  Documentos: 34
  Frecuencia total: 37
  Posting list:
    - doc_index=14422, job_id=3f39df6adf74b1119ced57864fba75982b8c246723024ec1788774112c9a7d23, freq=1, title=Laborer
    - doc_index=15849, job_id=2ab4ada1512f59548c43f1d3823693c803fd38979f802e35b068f33e64f29982, freq=1, title=LAUREATA\\O IN GIURISPRUDENZA/ECONOMIA o CONSULENTE D' IMPRESA
    - doc_index=18356, job_id=5f46afc2a364f40917cbb54b034c733501d49a54b16a14e445092a645a11b0c6, freq=1, title=T.S.U. en Electricidad
    - doc_index=21420, job_id=6dfd598d8f2efac8963923068a292d26c3bc60bbe00d4955949bcf5c54103216, freq=1, title=Laborer
    - doc_index=21431, job_id=c71e17a9e97a6d9cc6d773517987cb8bd0896c45fd3f63bf2c777cb6436e67f2, freq=1, title=Cement Driver

Término: 'aaa'
  Documentos: 25
  Frecuencia total: 32
  Posting list:
    - doc_index=7467, job_id=bda92e9fbaab5d71114392743bb6c82bdf5d75a437613f98f0ef48d9dcd4c603, freq=1, title=Pastry Chef de Partie
    - doc_index=9000, job_id=aa25238b7ed6a546b0cbc1cfe8bf359

## Paso 7: TF-IDF

In [7]:
# Construir matriz TF-IDF a partir del texto stemmed
vectorizer, tfidf_matrix = build_tfidf_matrix(df_corpus['stemmed'].tolist())

print(f"✅ Matriz TF-IDF construida")
print(f"   Dimensiones: {tfidf_matrix.shape[0]} documentos × {tfidf_matrix.shape[1]} términos")
print(f"   Sparsidad: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")

# Obtener los términos más frecuentes
feature_names = vectorizer.get_feature_names_out()
print(f"\n Primeros 20 términos del vocabulario:")
print(f"   {list(feature_names[:20])}")

# Estadísticas por documento
doc_term_counts = (tfidf_matrix > 0).sum(axis=1).A1  # Contar términos no-cero por documento
print(f"\n Estadísticas de términos por documento:")
print(f"   Promedio de términos únicos: {doc_term_counts.mean():.0f}")
print(f"   Máximo: {doc_term_counts.max()}")
print(f"   Mínimo: {doc_term_counts.min()}")

print(f"\n✅ df_corpus tiene {len(df_corpus)} documentos listos para recuperación")
print(f"\nEstructura final de df_corpus:")
print(df_corpus.columns.tolist())

✅ Matriz TF-IDF construida
   Dimensiones: 75695 documentos × 56999 términos
   Sparsidad: 99.92%

 Primeros 20 términos del vocabulario:
   ['aa', 'aaa', 'aaaradius', 'aaas', 'aac', 'aacp', 'aacsb', 'aact', 'aad', 'aadus', 'aae', 'aaeeo', 'aaeeoveteransdiscapacit', 'aaf', 'aah', 'aai', 'aailndi', 'aailndiam', 'aajax', 'aan']

 Estadísticas de términos por documento:
   Promedio de términos únicos: 44
   Máximo: 708
   Mínimo: 0

✅ df_corpus tiene 75695 documentos listos para recuperación

Estructura final de df_corpus:
['job_id', 'job_title', 'company', 'careers_required', 'text', 'clean_text', 'tokens', 'stemmed']


## Paso 7.1: Ejemplo de consultas TF-IDF con resultados numéricos

In [8]:
queries = [
    'inteligencia artificial',
    'ingeniería de software',
    'energía renovable',
    'análisis de datos',
    'diseño de sistemas electrónicos',
    'gestión de proyectos'
]

scores = score_queries_tfidf(vectorizer, tfidf_matrix, queries)

results = []
for query_idx, query in enumerate(queries):
    for doc_idx, score in enumerate(scores):
        results.append({
            'query': query,
            'doc_index': doc_idx,
            'score': score[query_idx],
            'job_title': df_corpus['job_title'].iloc[doc_idx],
            'company': df_corpus['company'].iloc[doc_idx],
            'preview': df_corpus['clean_text'].iloc[doc_idx][:120].replace('\n', ' ')
        })

results_df = pd.DataFrame(results)

# Mostrar top 3 documentos por consulta
for query in queries:
    print(f"\nConsulta: '{query}'")
    top_docs = results_df[results_df['query'] == query].nlargest(3, 'score')
    display(top_docs[['score', 'job_title', 'company', 'preview']])



Consulta: 'inteligencia artificial'


,score,job_title,company,preview
43621,0.474468,"Artificial Intelligence Sales Specialist III, ...",Google,s Grado o experiencia práctica equivalente 10...
47933,0.469508,Technology Analyst 2-Artificial Intelligence (...,StateJobsNY,Tecnology Analyst 2 inteligencia artificial Ba...
43441,0.449351,Strategic Artificial Intelligence Consultant -...,Business and Technology Solutions,Nuestro cliente en Rhode Island está buscando ...



Consulta: 'ingeniería de software'


,score,job_title,company,preview
124981,0.232339,Líder de equipo,Buffalo Wild Wings GO,miembros del equipo según lo solicitado Propor...
94292,0.209075,HVAC Heating and Cooling Systems Installer,Rellaire Smart Home Systems,Ya sea que recién esté comenzando o sea un ins...
119242,0.200060,Sr. Dir. Product Management - Artificial Intel...,UKG (Ultimate Kronos Group),No puedo esperar para apoyar lo que te dé un p...



Consulta: 'energía renovable'


...

,score,job_title,company,preview
276371,0.391171,Líder de equipo,Buffalo Wild Wings GO,miembros del equipo según lo solicitado Propor...
245682,0.352004,HVAC Heating and Cooling Systems Installer,Rellaire Smart Home Systems,Ya sea que recién esté comenzando o sea un ins...
270632,0.336826,Sr. Dir. Product Management - Artificial Intel...,UKG (Ultimate Kronos Group),No puedo esperar para apoyar lo que te dé un p...



Consulta: 'diseño de sistemas electrónicos'


,score,job_title,company,preview
352066,0.391171,Líder de equipo,Buffalo Wild Wings GO,miembros del equipo según lo solicitado Propor...
321377,0.352004,HVAC Heating and Cooling Systems Installer,Rellaire Smart Home Systems,Ya sea que recién esté comenzando o sea un ins...
346327,0.336826,Sr. Dir. Product Management - Artificial Intel...,UKG (Ultimate Kronos Group),No puedo esperar para apoyar lo que te dé un p...



Consulta: 'gestión de proyectos'


,score,job_title,company,preview
427761,0.391171,Líder de equipo,Buffalo Wild Wings GO,miembros del equipo según lo solicitado Propor...
397072,0.352004,HVAC Heating and Cooling Systems Installer,Rellaire Smart Home Systems,Ya sea que recién esté comenzando o sea un ins...
422022,0.336826,Sr. Dir. Product Management - Artificial Intel...,UKG (Ultimate Kronos Group),No puedo esperar para apoyar lo que te dé un p...


## Paso 8: Jaccard